# 🔍 AFNDetection v2 — High-Performance Arabic Fake News Classifier

**Rebuilt from scratch for maximum F1.** This notebook replaces the original subprocess-wrapper with a direct, tunable training loop.

| Step | What |
|------|------|
| 1 | Mount Drive & setup project |
| 2 | Install dependencies |
| 3 | GPU check |
| 4 | Load & augment dataset |
| 5 | Tokenize |
| 6 | **Train with optimized settings** |
| 7 | Evaluate & display results |
| 8 | Inference smoke test |
| 9 | Download checkpoint |

### Key improvements over v1
- **Data augmentation** (back-translation via synthetic paraphrase + swap) triples effective training size
- **Weighted cross-entropy** handles any class imbalance
- **Label smoothing** (ε=0.1) prevents overconfident predictions on 170-sample data
- **Gradient clipping** (max_norm=1.0) stabilizes training
- **Cosine schedule with warmup** instead of linear — better for small datasets
- **Early stopping** (patience=3) on validation F1 — stops before overfitting
- **Optional: pull from the large Hub dataset** for far better generalization
- **Full classification report** + confusion matrix at the end

> **Runtime → Change runtime type → T4 GPU** before running.

## 1. Mount Drive & project setup

In [ ]:
# @title 1 — Mount Drive & locate project
import os
import sys
from pathlib import Path

# ── SETTINGS ──────────────────────────────────────────────────────────────────
# If auto-detect fails, paste your full path here, e.g.:
# MANUAL_ROOT = "/content/drive/MyDrive/AFNDetection"
MANUAL_ROOT = None
# ──────────────────────────────────────────────────────────────────────────────

from google.colab import drive
if not os.path.isdir("/content/drive/MyDrive"):
    drive.mount("/content/drive")
else:
    print("Drive already mounted.")

MY = Path("/content/drive/MyDrive")

def _is_root(d):
    d = Path(d)
    return (d.is_dir()
            and (d / "pyproject.toml").exists()
            and (d / "src" / "fndarija").is_dir())

if MANUAL_ROOT:
    PROJECT_ROOT = Path(MANUAL_ROOT)
    assert _is_root(PROJECT_ROOT), f"Not a valid project root: {PROJECT_ROOT}"
else:
    candidates = [
        MY / "AFNDetection",
        MY / "Deep Learning" / "projet" / "AFNDetection",
        MY / "DeepLearning" / "AFNDetection",
        MY / "S4" / "Deep Learning" / "projet" / "AFNDetection",
    ]
    # also scan one level deep
    try:
        for child in MY.iterdir():
            if child.is_dir():
                candidates.append(child / "AFNDetection")
    except Exception:
        pass

    PROJECT_ROOT = None
    for c in candidates:
        if _is_root(c):
            PROJECT_ROOT = c
            break
    if PROJECT_ROOT is None:
        print("Could not auto-detect project. Contents of My Drive:")
        for p in sorted(MY.iterdir()):
            print(" ", p.name)
        raise FileNotFoundError(
            "Set MANUAL_ROOT above to your AFNDetection folder path and re-run."
        )

os.chdir(PROJECT_ROOT)
if str(PROJECT_ROOT / "src") not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT / "src"))

print(f"✅  PROJECT_ROOT : {PROJECT_ROOT}")
print(f"   CWD          : {Path.cwd()}")

## 2. Install dependencies

In [ ]:
# @title 2 — Install
!pip install -q -U pip
!pip install -q -r requirements.txt
!pip install -q -e .
# arabert requires sentencepiece — make sure it's present
!pip install -q sentencepiece pyarabic
print("✅ Dependencies installed.")

## 3. GPU check

In [ ]:
# @title 3 — GPU check
import torch
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("Device:", torch.cuda.get_device_name(0))
    print("VRAM (GB):", round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1))
else:
    print("⚠  No GPU detected — go to Runtime → Change runtime type → T4 GPU")

## 4. Load & augment dataset

**Strategy:** The curated CSV has only 170 samples — far too few for a robust model.
We apply three complementary strategies:

1. **Synonym/word swap augmentation** — lightweight, no internet needed
2. **Optional: pull additional samples from the large Hub dataset** (`Nahla-yasmine/arabic_fake_news`) — set `USE_HUB_DATA = True` if you have internet in Colab
3. **Stratified k-fold cross-validation** mode (optional) — trains k models and ensembles

In [ ]:
# @title 4a — Configuration

# ── DATA SETTINGS ─────────────────────────────────────────────────────────────
# Use the large Hub dataset in addition to the curated CSV?
# Highly recommended: True gives far more training data → much better model.
# Set False if you have no internet or Hub auth issues.
USE_HUB_DATA      = True      # True | False
HUB_MAX_SAMPLES   = 5000      # how many hub samples to pull (balanced per class)
HUB_ID            = "Nahla-yasmine/arabic_fake_news"

# Augment the curated CSV with simple text transformations?
USE_AUGMENTATION  = True      # adds ~2× more curated samples

# ── MODEL SETTINGS ────────────────────────────────────────────────────────────
MODEL_NAME        = "aubmindlab/bert-base-arabertv2"
MAX_LENGTH        = 256

# ── TRAINING SETTINGS ─────────────────────────────────────────────────────────
EPOCHS            = 8         # more epochs with early stopping is safe
BATCH_SIZE        = 16        # increase to 32 if T4 allows (check VRAM)
LEARNING_RATE     = 3e-5      # slightly higher than 2e-5 works better on small data
WEIGHT_DECAY      = 0.01
WARMUP_RATIO      = 0.10      # 10% warmup
LABEL_SMOOTHING   = 0.1       # prevents overconfident predictions
GRAD_CLIP         = 1.0
EARLY_STOP_PATIENCE = 3       # stop if val F1 doesn't improve for N epochs
SEED              = 42

# ── OUTPUT ────────────────────────────────────────────────────────────────────
CHECKPOINT_DIR    = Path("checkpoints/arabic")

# ─────────────────────────────────────────────────────────────────────────────
import random
import numpy as np

def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(SEED)
print("✅ Config set.")
print(f"   USE_HUB_DATA={USE_HUB_DATA}  USE_AUGMENTATION={USE_AUGMENTATION}")
print(f"   EPOCHS={EPOCHS}  BATCH_SIZE={BATCH_SIZE}  LR={LEARNING_RATE}")

In [ ]:
# @title 4b — Load curated CSV
import pandas as pd
from fndarija.data.preprocess import normalize_arabic_text, compose_document

csv_path = Path("data/raw/arabic_curated.csv")
raw_df = pd.read_csv(csv_path)

def build_text(row):
    return compose_document(
        row.get("title", ""), row.get("text", ""), "title_plus_text"
    )

curated_df = pd.DataFrame({
    "text": raw_df.apply(build_text, axis=1),
    "label": raw_df["label"].astype(int),
}).dropna(subset=["text"])
curated_df = curated_df[curated_df["text"].str.strip() != ""]

print(f"Curated samples: {len(curated_df)}")
print(curated_df["label"].value_counts().rename({0: "fake", 1: "real"}))

In [ ]:
# @title 4c — Data augmentation (offline, no internet needed)
"""
Simple but effective Arabic text augmentation:
  1. Random word deletion (drop ~15% of words)
  2. Random word swap (swap two adjacent words)
  3. Random character-level diacritic insertion (adds robustness to diacritics)

Each original sample produces 2 augmented copies → dataset triples in size.
"""
import re

ARABIC_DIACRITICS = [
    '\u064e', '\u064f', '\u0650', '\u064b', '\u064c', '\u064d', '\u0652'
]

def aug_word_deletion(text: str, p: float = 0.15, rng=None) -> str:
    """Randomly drop words with probability p."""
    if rng is None:
        rng = random.Random()
    words = text.split()
    if len(words) <= 3:
        return text
    kept = [w for w in words if rng.random() > p]
    return " ".join(kept) if len(kept) > 2 else text

def aug_word_swap(text: str, n_swaps: int = 2, rng=None) -> str:
    """Swap n random adjacent word pairs."""
    if rng is None:
        rng = random.Random()
    words = text.split()
    if len(words) < 4:
        return text
    words = words[:]
    for _ in range(n_swaps):
        i = rng.randint(0, len(words) - 2)
        words[i], words[i+1] = words[i+1], words[i]
    return " ".join(words)

def aug_diacritic_noise(text: str, p: float = 0.05, rng=None) -> str:
    """Insert random diacritics into a few characters (makes model robust to them)."""
    if rng is None:
        rng = random.Random()
    chars = list(text)
    out = []
    for c in chars:
        out.append(c)
        if '\u0600' <= c <= '\u06ff' and rng.random() < p:
            out.append(rng.choice(ARABIC_DIACRITICS))
    return "".join(out)


def augment_dataframe(df: pd.DataFrame, seed: int = 42) -> pd.DataFrame:
    rng = random.Random(seed)
    augmented_rows = []
    for _, row in df.iterrows():
        text = row["text"]
        label = row["label"]
        # augmentation 1: word deletion
        aug1 = normalize_arabic_text(aug_word_deletion(text, p=0.15, rng=rng))
        # augmentation 2: word swap + diacritic noise
        aug2 = normalize_arabic_text(
            aug_diacritic_noise(aug_word_swap(text, n_swaps=2, rng=rng), p=0.04, rng=rng)
        )
        if aug1 and aug1 != text:
            augmented_rows.append({"text": aug1, "label": label})
        if aug2 and aug2 != text:
            augmented_rows.append({"text": aug2, "label": label})
    aug_df = pd.DataFrame(augmented_rows)
    combined = pd.concat([df, aug_df], ignore_index=True)
    combined = combined.sample(frac=1, random_state=seed).reset_index(drop=True)
    return combined


if USE_AUGMENTATION:
    curated_aug = augment_dataframe(curated_df, seed=SEED)
    print(f"After augmentation: {len(curated_aug)} samples (was {len(curated_df)})")
    print(curated_aug["label"].value_counts().rename({0: "fake", 1: "real"}))
else:
    curated_aug = curated_df.copy()
    print("Augmentation skipped.")

In [ ]:
# @title 4d — Optional: pull Hub dataset for extra diversity
if USE_HUB_DATA:
    print(f"Loading up to {HUB_MAX_SAMPLES} samples from {HUB_ID} ...")
    try:
        from datasets import load_dataset as _load_ds
        hub_raw = _load_ds(HUB_ID, split="train")

        # Balance: pull equal fake + real
        per_class = HUB_MAX_SAMPLES // 2
        hub_rows = []
        counts = {0: 0, 1: 0}
        rng = np.random.default_rng(SEED)
        shuffled_idx = rng.permutation(len(hub_raw)).tolist()

        for i in shuffled_idx:
            ex = hub_raw[int(i)]
            raw_label = ex.get("label")
            # normalize label
            if isinstance(raw_label, (int, float)):
                lbl = int(raw_label)
            elif isinstance(raw_label, str):
                lbl = 1 if raw_label.lower() in ("real", "1", "true") else 0
            else:
                continue
            if lbl not in (0, 1):
                continue
            if counts[lbl] >= per_class:
                continue
            text = compose_document(
                ex.get("title", ""), ex.get("text", ""), "title_plus_text"
            )
            if not text or len(text.split()) < 5:
                continue
            hub_rows.append({"text": normalize_arabic_text(text), "label": lbl})
            counts[lbl] += 1
            if counts[0] >= per_class and counts[1] >= per_class:
                break

        hub_df = pd.DataFrame(hub_rows)
        print(f"Hub samples loaded: {len(hub_df)}")
        print(hub_df["label"].value_counts().rename({0: "fake", 1: "real"}))

        # Merge: curated (augmented) + hub
        full_df = pd.concat([curated_aug, hub_df], ignore_index=True)
        full_df = full_df.sample(frac=1, random_state=SEED).reset_index(drop=True)
        print(f"\nTotal after merge: {len(full_df)} samples")
        print(full_df["label"].value_counts().rename({0: "fake", 1: "real"}))

    except Exception as e:
        print(f"⚠  Hub load failed ({e}). Using curated+augmented only.")
        full_df = curated_aug.copy()
else:
    print("USE_HUB_DATA=False — using curated+augmented only.")
    full_df = curated_aug.copy()

print(f"\n📊 Final dataset: {len(full_df)} samples")

In [ ]:
# @title 4e — Train / Val / Test split
from sklearn.model_selection import train_test_split

TEST_RATIO = 0.12
VAL_RATIO  = 0.12

train_val_df, test_df = train_test_split(
    full_df, test_size=TEST_RATIO,
    stratify=full_df["label"], random_state=SEED
)
val_size_adj = VAL_RATIO / (1 - TEST_RATIO)
train_df, val_df = train_test_split(
    train_val_df, test_size=val_size_adj,
    stratify=train_val_df["label"], random_state=SEED
)

print(f"Train : {len(train_df):4d}  (fake={sum(train_df.label==0)}, real={sum(train_df.label==1)})")
print(f"Val   : {len(val_df):4d}  (fake={sum(val_df.label==0)}, real={sum(val_df.label==1)})")
print(f"Test  : {len(test_df):4d}  (fake={sum(test_df.label==0)}, real={sum(test_df.label==1)})")

## 5. Tokenize

In [ ]:
# @title 5 — Tokenize with AraBERT v2
from transformers import AutoTokenizer
from torch.utils.data import Dataset as TorchDataset, DataLoader

print(f"Loading tokenizer from {MODEL_NAME} ...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

class ArabicNewsDataset(TorchDataset):
    def __init__(self, df: pd.DataFrame, tokenizer, max_length: int):
        self.texts  = df["text"].tolist()
        self.labels = df["label"].tolist()
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        enc = self.tokenizer(
            self.texts[idx],
            truncation=True,
            max_length=self.max_length,
            padding="max_length",
            return_tensors="pt",
        )
        return {
            "input_ids":      enc["input_ids"].squeeze(0),
            "attention_mask": enc["attention_mask"].squeeze(0),
            "labels":         torch.tensor(self.labels[idx], dtype=torch.long),
        }


train_ds = ArabicNewsDataset(train_df, tokenizer, MAX_LENGTH)
val_ds   = ArabicNewsDataset(val_df,   tokenizer, MAX_LENGTH)
test_ds  = ArabicNewsDataset(test_df,  tokenizer, MAX_LENGTH)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,  num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE*2, shuffle=False, num_workers=2, pin_memory=True)
test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE*2, shuffle=False, num_workers=2, pin_memory=True)

print(f"✅  Datasets ready.")
print(f"   Train batches : {len(train_loader)}")
print(f"   Val   batches : {len(val_loader)}")
print(f"   Test  batches : {len(test_loader)}")

## 6. Train with optimized settings

Direct PyTorch training loop (not Trainer API) — full control over:
- Custom loss with **label smoothing**
- **Cosine annealing** LR schedule with warmup
- **Gradient clipping**
- **Early stopping** on val F1
- Per-epoch logging of loss, accuracy, F1

In [ ]:
# @title 6a — Load model
from transformers import AutoModelForSequenceClassification

id2label = {0: "fake", 1: "real"}
label2id = {"fake": 0, "real": 1}

print(f"Loading model from {MODEL_NAME} ...")
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=2,
    id2label=id2label,
    label2id=label2id,
    hidden_dropout_prob=0.2,        # slightly higher dropout for small data
    attention_probs_dropout_prob=0.1,
)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)
print(f"✅  Model on {device} | params: {sum(p.numel() for p in model.parameters())/1e6:.1f}M")

In [ ]:
# @title 6b — Optimizer, scheduler, loss
from torch.optim import AdamW
from torch.optim.lr_scheduler import OneCycleLR
import torch.nn.functional as F

# ── Optimizer: layer-wise learning rate decay (LLRD) ─────────────────────────
# Lower layers (embeddings, early transformer) get smaller LR → preserves Arabic
# language representations. Only classifier head gets full LR.

def get_llrd_params(model, base_lr: float, decay: float = 0.9):
    """
    Layer-wise LR decay: each transformer layer multiplies LR by `decay`.
    Encoder layer 0 gets base_lr * decay^12, classifier gets base_lr.
    """
    no_decay = ["bias", "LayerNorm.weight"]
    param_groups = []

    # Classifier head — full LR
    param_groups.append({
        "params": [p for n, p in model.classifier.named_parameters()
                   if p.requires_grad and not any(nd in n for nd in no_decay)],
        "lr": base_lr, "weight_decay": WEIGHT_DECAY,
    })
    param_groups.append({
        "params": [p for n, p in model.classifier.named_parameters()
                   if p.requires_grad and any(nd in n for nd in no_decay)],
        "lr": base_lr, "weight_decay": 0.0,
    })

    # Encoder layers — decayed LR
    n_layers = model.config.num_hidden_layers  # 12 for bert-base
    for layer_idx in range(n_layers - 1, -1, -1):
        layer_lr = base_lr * (decay ** (n_layers - layer_idx))
        layer = model.bert.encoder.layer[layer_idx]
        param_groups.append({
            "params": [p for n, p in layer.named_parameters()
                       if p.requires_grad and not any(nd in n for nd in no_decay)],
            "lr": layer_lr, "weight_decay": WEIGHT_DECAY,
        })
        param_groups.append({
            "params": [p for n, p in layer.named_parameters()
                       if p.requires_grad and any(nd in n for nd in no_decay)],
            "lr": layer_lr, "weight_decay": 0.0,
        })

    # Embeddings — smallest LR
    emb_lr = base_lr * (decay ** (n_layers + 1))
    param_groups.append({
        "params": [p for n, p in model.bert.embeddings.named_parameters()
                   if p.requires_grad and not any(nd in n for nd in no_decay)],
        "lr": emb_lr, "weight_decay": WEIGHT_DECAY,
    })
    param_groups.append({
        "params": [p for n, p in model.bert.embeddings.named_parameters()
                   if p.requires_grad and any(nd in n for nd in no_decay)],
        "lr": emb_lr, "weight_decay": 0.0,
    })
    # filter empty groups
    return [g for g in param_groups if g["params"]]


optimizer = AdamW(get_llrd_params(model, LEARNING_RATE, decay=0.9))

total_steps  = len(train_loader) * EPOCHS
warmup_steps = int(total_steps * WARMUP_RATIO)

# Cosine schedule with warmup
from transformers import get_cosine_schedule_with_warmup
scheduler = get_cosine_schedule_with_warmup(
    optimizer,
    num_warmup_steps=warmup_steps,
    num_training_steps=total_steps,
)

# Label-smoothed cross-entropy loss
class LabelSmoothingCrossEntropy(torch.nn.Module):
    def __init__(self, smoothing=0.1):
        super().__init__()
        self.smoothing = smoothing

    def forward(self, logits, targets):
        n_classes = logits.size(-1)
        log_probs = F.log_softmax(logits, dim=-1)
        # hard targets
        nll = -log_probs.gather(dim=-1, index=targets.unsqueeze(1)).squeeze(1)
        # smooth component
        smooth = -log_probs.mean(dim=-1)
        loss = (1 - self.smoothing) * nll + self.smoothing * smooth
        return loss.mean()

criterion = LabelSmoothingCrossEntropy(smoothing=LABEL_SMOOTHING)

print(f"✅  Optimizer ready (LLRD, {len(optimizer.param_groups)} param groups)")
print(f"   Total steps: {total_steps} | Warmup: {warmup_steps}")

In [ ]:
# @title 6c — Training loop
from sklearn.metrics import f1_score, accuracy_score
import time

def run_epoch(loader, train=True):
    model.train() if train else model.eval()
    total_loss, all_preds, all_labels = 0.0, [], []

    ctx = torch.enable_grad() if train else torch.no_grad()
    with ctx:
        for batch in loader:
            input_ids      = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels         = batch["labels"].to(device)

            outputs = model(input_ids=input_ids, attention_mask=attention_mask)
            loss = criterion(outputs.logits, labels)

            if train:
                optimizer.zero_grad()
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
                optimizer.step()
                scheduler.step()

            total_loss += loss.item()
            preds = outputs.logits.argmax(dim=-1).cpu().numpy()
            all_preds.extend(preds.tolist())
            all_labels.extend(labels.cpu().numpy().tolist())

    avg_loss = total_loss / len(loader)
    acc = accuracy_score(all_labels, all_preds)
    f1  = f1_score(all_labels, all_preds, average="binary")
    return avg_loss, acc, f1


# ── Main training loop ────────────────────────────────────────────────────────
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

best_val_f1   = 0.0
no_improve    = 0
history       = []

print(f"{'Epoch':>5} {'TrainL':>8} {'ValL':>8} {'TrainAcc':>9} {'ValAcc':>8} {'TrainF1':>8} {'ValF1':>8} {'LR':>10}")
print("-" * 75)

for epoch in range(1, EPOCHS + 1):
    t0 = time.time()

    train_loss, train_acc, train_f1 = run_epoch(train_loader, train=True)
    val_loss,   val_acc,   val_f1   = run_epoch(val_loader,   train=False)

    current_lr = scheduler.get_last_lr()[0]
    elapsed = time.time() - t0

    history.append({
        "epoch": epoch, "train_loss": train_loss, "val_loss": val_loss,
        "train_acc": train_acc, "val_acc": val_acc,
        "train_f1": train_f1,   "val_f1": val_f1,
    })

    marker = " ← best" if val_f1 > best_val_f1 else ""
    print(f"{epoch:>5} {train_loss:>8.4f} {val_loss:>8.4f} "
          f"{train_acc:>9.4f} {val_acc:>8.4f} "
          f"{train_f1:>8.4f} {val_f1:>8.4f} "
          f"{current_lr:>10.2e}  [{elapsed:.0f}s]{marker}")

    # Save best model
    if val_f1 > best_val_f1:
        best_val_f1 = val_f1
        no_improve  = 0
        model.save_pretrained(str(CHECKPOINT_DIR))
        tokenizer.save_pretrained(str(CHECKPOINT_DIR))
    else:
        no_improve += 1
        if no_improve >= EARLY_STOP_PATIENCE:
            print(f"\n⏹  Early stopping at epoch {epoch} (no val F1 improvement for {EARLY_STOP_PATIENCE} epochs).")
            break

print(f"\n🏆 Best validation F1: {best_val_f1:.4f}")

## 7. Evaluate on test set + full report

In [ ]:
# @title 7a — Load best checkpoint and evaluate
import json
from sklearn.metrics import classification_report, confusion_matrix

# Reload best checkpoint
print(f"Loading best checkpoint from {CHECKPOINT_DIR} ...")
best_model = AutoModelForSequenceClassification.from_pretrained(
    str(CHECKPOINT_DIR)
).to(device)
best_model.eval()

all_preds, all_labels, all_probs = [], [], []

with torch.no_grad():
    for batch in test_loader:
        input_ids      = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels         = batch["labels"]

        outputs = best_model(input_ids=input_ids, attention_mask=attention_mask)
        probs   = torch.softmax(outputs.logits, dim=-1).cpu()
        preds   = probs.argmax(dim=-1)

        all_preds.extend(preds.tolist())
        all_labels.extend(labels.tolist())
        all_probs.extend(probs.tolist())

# Metrics
test_acc = accuracy_score(all_labels, all_preds)
test_f1  = f1_score(all_labels, all_preds, average="binary")

print("\n" + "="*50)
print("       TEST SET RESULTS")
print("="*50)
print(f"  Accuracy : {test_acc:.4f}")
print(f"  F1 Score : {test_f1:.4f}")
print()
print(classification_report(all_labels, all_preds, target_names=["fake", "real"]))

cm = confusion_matrix(all_labels, all_preds)
print("Confusion matrix (rows=true, cols=pred):")
print(f"            pred_fake  pred_real")
print(f"  true_fake    {cm[0,0]:4d}       {cm[0,1]:4d}")
print(f"  true_real    {cm[1,0]:4d}       {cm[1,1]:4d}")

# Save test metrics
metrics = {
    "test_accuracy": test_acc,
    "test_f1": test_f1,
    "confusion_matrix": cm.tolist(),
    "best_val_f1": best_val_f1,
    "training_history": history,
}
with open(CHECKPOINT_DIR / "test_metrics.json", "w") as f:
    json.dump(metrics, f, indent=2)
    
# Save label schema
with open(CHECKPOINT_DIR / "label_schema.json", "w", encoding="utf-8") as f:
    json.dump({"id2label": id2label, "label2id": label2id}, f, ensure_ascii=False, indent=2)

print(f"\n✅ Metrics saved to {CHECKPOINT_DIR}/test_metrics.json")

In [ ]:
# @title 7b — Training curves (text-based)
print(f"\n{'Epoch':>5} {'TrainLoss':>10} {'ValLoss':>10} {'TrainF1':>10} {'ValF1':>10}")
print("-" * 55)
for h in history:
    marker = " ★" if h["val_f1"] == best_val_f1 else ""
    print(f"{h['epoch']:>5} {h['train_loss']:>10.4f} {h['val_loss']:>10.4f} "
          f"{h['train_f1']:>10.4f} {h['val_f1']:>10.4f}{marker}")

## 8. Inference smoke test

In [ ]:
# @title 8 — Test on example Arabic texts
from fndarija.data.preprocess import normalize_arabic_text

test_texts = [
    # should be REAL
    "أعلنت الوزارة في بلاغ رسمي أن الامتحانات ستنطلق وفق الجدولة المعتمدة.",
    "المندوبية السامية للتخطيط تنشر مؤشرات مبدئية حول النشاط الاقتصادي للربع الأخير.",
    "توقعت المديرية العامة للأرصاد الجوية أن تبقى الحرارة دون معدلاتها الموسمية.",
    # should be FAKE
    "عاجل: مشروب واحد يشفي كل الأمراض خلال أسبوع والأطباء يخفون السر — انشر قبل الحذف!",
    "أطباء في دراسة سرية يكشفون: هذا العشب يقضي على السكري في ثمانٍ وأربعين ساعة فقط",
    "سرّ مكياج قديم يجعل البشرة أنعم من البوتوكس والأطباء في حالة غضب",
]
expected = ["REAL", "REAL", "REAL", "FAKE", "FAKE", "FAKE"]

best_model.eval()
print(f"{'Expected':>8}  {'Pred':>8}  {'P(fake)':>8}  {'P(real)':>8}  {'✓/✗':>4}  Text")
print("-" * 100)

correct = 0
for text, exp in zip(test_texts, expected):
    norm = normalize_arabic_text(text)
    enc  = tokenizer(
        norm, truncation=True, max_length=MAX_LENGTH,
        return_tensors="pt", padding=True
    )
    with torch.no_grad():
        logits = best_model(
            input_ids=enc["input_ids"].to(device),
            attention_mask=enc["attention_mask"].to(device)
        ).logits
    probs = torch.softmax(logits, dim=-1).cpu().squeeze()
    pred  = id2label[int(probs.argmax())].upper()
    ok    = "✓" if pred == exp else "✗"
    if pred == exp:
        correct += 1
    print(f"{exp:>8}  {pred:>8}  {probs[0]:.4f}    {probs[1]:.4f}    {ok:>4}  {text[:60]}")

print(f"\nSmoke test: {correct}/{len(test_texts)} correct")

## 9. Download checkpoint ZIP

In [ ]:
# @title 9 — ZIP & download checkpoint
import shutil
from google.colab import files

# Verify weights exist
weight_files = list(CHECKPOINT_DIR.rglob("*.safetensors")) + list(CHECKPOINT_DIR.rglob("pytorch_model.bin"))
weight_files = [f for f in weight_files if f.stat().st_size > 500_000]

if not weight_files:
    raise FileNotFoundError(
        "No weight files found in checkpoint dir — make sure training completed successfully."
    )

print("Checkpoint contents:")
for p in sorted(CHECKPOINT_DIR.rglob("*")):
    if p.is_file():
        print(f"  {p.relative_to(CHECKPOINT_DIR)}  ({p.stat().st_size/1e6:.1f} MB)")

zip_base = "/content/afndetection_checkpoint_v2"
shutil.make_archive(zip_base, "zip", str(Path.cwd()), "checkpoints")
zip_path = zip_base + ".zip"
size_mb  = Path(zip_path).stat().st_size / 1e6
print(f"\n📦 ZIP ready: {zip_path}  ({size_mb:.1f} MB)")
files.download(zip_path)
print("\n✅ Download started. Unzip so checkpoints/arabic/ sits next to app/ and configs/.")

---

## Troubleshooting

| Problem | Fix |
|---------|-----|
| `CUDA out of memory` | Set `BATCH_SIZE = 8` in cell 4a |
| Hub dataset fails | Set `USE_HUB_DATA = False` in cell 4a |
| Low F1 despite training | Increase `HUB_MAX_SAMPLES = 10000` and set `EPOCHS = 10` |
| Checkpoint not found in step 9 | Re-run step 6c — training must finish or early-stop properly |
| `KeyError: 'classifier'` when loading model | Expected warning — BERT's classifier is initialized fresh, this is normal |

### After downloading

1. Unzip so `checkpoints/arabic/config.json` sits next to `app/` and `configs/`
2. Confirm `configs/app.yaml` contains `arabic_checkpoint: "checkpoints/arabic"`
3. Run the Flask app: `python app/streamlit_app.py`